# *Cannabis sativa* metabolic reconstruction in Terpedia

This Colab notebook reproduces the catalog, reachability, and hypothesis results reported in the accompanying paper. It reads the versioned Terpedia Knowledge artifacts and independently reruns the reaction hypergraph closure.

In [ ]:
from pathlib import Path
import json, subprocess, pandas as pd
REPO = Path('/content/terpedia-knowledge')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Terpedia/terpedia-knowledge.git', str(REPO)], check=True)
REPORTS = REPO / 'data' / 'reports'
NETWORK = REPO / 'data' / 'semantic' / 'cannabis-sativa-metabolic-network.json'
network = json.loads(NETWORK.read_text())
reachability = json.loads((REPORTS / 'cannabis-sativa-metabolic-reachability.json').read_text())
hypotheses = json.loads((REPORTS / 'cannabis-sativa-metabolic-hypotheses.json').read_text())
catalog = json.loads((REPORTS / 'cannabis-sativa-metabolic-catalog.json').read_text())
print(network['provenance']['datasetVersion'])

## Methods

Reactions are directed hyperedges. A reaction fires only when every input metabolite is reachable. The denominator is every unique ChEBI entity appearing as a reaction reactant. The medium proxy seeds glucose forms, inorganic nitrogen, phosphate, sulfate, common ions, molybdate, water, proton, oxygen, and carbon dioxide; unavailable ingredients remain explicit in the source report.

In [ ]:
reaction_ids = {e['id'] for e in network['entities'] if e.get('type') == 'biochemical_reaction'}
reactions = {rid: {'reactants': set(), 'products': set(), 'directions': set()} for rid in reaction_ids}
for statement in network['statements']:
    reaction = reactions.get(statement.get('subjectId'))
    if reaction is None: continue
    if statement['predicate'] == 'has_reactant': reaction['reactants'].add(statement['objectEntityId'])
    elif statement['predicate'] == 'has_product': reaction['products'].add(statement['objectEntityId'])
    elif statement['predicate'] == 'physiological_direction_left_to_right': reaction['directions'].add('forward')
    elif statement['predicate'] == 'physiological_direction_right_to_left': reaction['directions'].add('reverse')
reactions = {k: v for k, v in reactions.items() if v['reactants'] and v['products']}
seed_ids = set(reachability['medium']['resolvedSeedEntityIds'])
def expand(mode):
    available = set(seed_ids)
    while True:
        additions = set()
        for reaction in reactions.values():
            directions = ['forward', 'reverse'] if mode == 'all-reactions-reversible' else list(reaction['directions'])
            if mode == 'curated-directions-plus-unknown-reversible' and not directions: directions = ['forward', 'reverse']
            for direction in directions:
                inputs = reaction['reactants'] if direction == 'forward' else reaction['products']
                outputs = reaction['products'] if direction == 'forward' else reaction['reactants']
                if inputs.issubset(available): additions.update(outputs)
        new = additions - available
        if not new: break
        available.update(new)
    denominator = {m for reaction in reactions.values() for m in reaction['reactants']}
    return len(available & denominator), len(denominator)
computed = {mode: expand(mode) for mode in reachability['modes']}
assert all(computed[mode][0] == result['reachableReactants'] and computed[mode][1] == result['reactantUniverse'] for mode, result in reachability['modes'].items())
print('Independent closure agrees with stored report.')

## Results

In [ ]:
reachability_table = pd.DataFrame([
    {'direction_mode': mode, 'reachable_reactants': result['reachableReactants'], 'denominator': result['reactantUniverse'], 'percent': result['reactantReachabilityPercent']}
    for mode, result in reachability['modes'].items()
])
display(reachability_table)
display(pd.DataFrame([
    {'measure': 'proteins', 'value': catalog['referenceProteome']['proteinCount']},
    {'measure': 'exact-EC proteins', 'value': catalog['enzymeAnnotation']['proteinsWithExactEc']},
    {'measure': 'Rhea reactions', 'value': reachability['network']['reactions']},
    {'measure': 'catalog ChEBI participants', 'value': catalog['reactionParticipantsAndBalance']['uniqueChebiParticipants']},
    {'measure': 'reactant denominator', 'value': reachability['network']['reactantUniverse']},
    {'measure': 'transporter candidates', 'value': catalog['compartmentAndTransport']['transporterCandidates']},
]))

In [ ]:
hypothesis_table = pd.DataFrame([
    {'hypothesis_class': 'unresolved exact EC', 'count': hypotheses['summary']['unresolvedExactEcHypotheses']},
    {'hypothesis_class': 'missing producer', 'count': hypotheses['summary']['missingProducerHypotheses']},
    {'hypothesis_class': 'blocked known reaction', 'count': hypotheses['summary']['blockedKnownReactionHypotheses']},
    {'hypothesis_class': 'blocked with candidate enzyme', 'count': hypotheses['summary']['hypothesesWithCandidateEnzyme']},
])
display(hypothesis_table)
import matplotlib.pyplot as plt
hypothesis_table.set_index('hypothesis_class')['count'].sort_values().plot(kind='barh', color='#2F6B4F', figsize=(8, 4))
plt.xlabel('Hypothesis count'); plt.title('Cannabis reconstruction gaps'); plt.tight_layout(); plt.show()

## Interpretation

The 3.13% reversible upper bound is a topology diagnostic, not a claim that Cannabis lacks metabolism. The primary missing layers are producer reactions, precursor/cofactor closure, transport, compartment assignments, exchange, biomass, thermodynamics, and Boolean gene–protein–reaction logic. CannabisDB compound and pathway associations provide orthogonal chemical context but do not establish stoichiometric reaction flux.